In [0]:
# dbutils.library.restartPython()

In [0]:
########################
# Active Transformation and Default Mapping Test Coverage Report
# Thin wrapper - all rendering logic lives in cross_coverage_report.py
########################

REPORT_VERSION = "v18"

# Changelog:
#   v18 (2026-04-29) — default row order now groups all tests for the same
#                      field together (sort key: from_state, field, verdict,
#                      test_name). Click a column header to override.
#   v17 (2026-04-29) — matrix rows are per-test, not per-field group.
#                      Added "FieldGroup" and "Test" columns; left-3 columns
#                      have fixed widths so the matrix fits on screen.
#                      Swapped chain_order to "...ended, remitted" so column
#                      order matches the live chain.
#   v16 — previous release.

# --- What to report on ---
runs_table = "test_automation_runs2"
results_table = "test_automation_results2"

# States (in chain order) the report understands. Edit freely.
chain_order = [
    "paymentPending",
    "appealSubmitted",
    "awaitingRespondentEvidence(a)",
    "awaitingRespondentEvidence(b)",
    "caseUnderReview",
    "reasonsForAppealSubmitted",
    "listing",
    "prepareForHearing",
    "decision",
    "decided(a)",
    "ftpaSubmitted(a)",
    "decided(b)",
    "ftpaSubmitted(b)",
    "ftpaDecided",
    "ended",
    "remitted",
]

# If set, only the listed states are included. Empty = every state in chain_order.
filter_states = []

# --- Field-matrix column widths (px). Tweak to fit your screen. ---
from_state_col_width_px   = 100
field_col_width_px        = 91
field_col_min_width_px    = 56
matrix_compact_font_px    = 10
heatmap_cell_min_width_px = 18

# --- Force reload so Databricks never serves a stale cached version ---
import sys, os
for _mod in list(sys.modules.keys()):
    if 'cross_coverage_report' in _mod:
        del sys.modules[_mod]


from cross_coverage_report import build_and_display

build_and_display(
    spark=spark,
    displayHTML=displayHTML,
    runs_table=runs_table,
    results_table=results_table,
    chain_order=chain_order,
    filter_states=filter_states,
    from_state_col_width_px=from_state_col_width_px,
    field_col_width_px=field_col_width_px,
    field_col_min_width_px=field_col_min_width_px,
    matrix_compact_font_px=matrix_compact_font_px,
    heatmap_cell_min_width_px=heatmap_cell_min_width_px,
    report_version=REPORT_VERSION,
)

In [0]:
# ----------------------------------------------------------------------
# A. The 3 AS init failure messages, per chain-child run_state
#    -> shows the column name(s) the select couldn't resolve
# ----------------------------------------------------------------------
print("=" * 90)
print("A. AS init-failure messages by run_state (latest run only)")
print("=" * 90)
init_field_set = ["DefaultMapping", "payment", "remission"]
init_errors = (latest.alias("r")
    .join(results.alias("t"), on="run_id", how="left")
    .filter((F.col("t.test_from_state") == "appealSubmitted")
            & (F.col("t.test_field").isin(init_field_set)))
    .select(F.col("r.state_under_test").alias("run_state"),
            F.col("t.test_field").alias("init"),
            F.substring("t.message", 1, 240).alias("message_240"))
    .orderBy("run_state", "init"))
init_errors.show(60, truncate=False)

# ----------------------------------------------------------------------
# B. Which columns are missing? Pull just the column names suggested
#    by [UNRESOLVED_COLUMN.WITH_SUGGESTION] errors.
#    Spark's error is roughly:
#      [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function
#      parameter with name `xxx` cannot be resolved. Did you mean ...
# ----------------------------------------------------------------------
print("=" * 90)
print("B. Missing-column names extracted from messages")
print("=" * 90)
missing = (init_errors
    .withColumn("missing_col", F.regexp_extract("message_240",
        r"with name `([^`]+)` cannot be resolved", 1))
    .filter("missing_col != ''")
    .select("run_state", "init", "missing_col")
    .distinct()
    .orderBy("run_state", "init"))
missing.show(60, truncate=False)

In [0]:
#Load Config and Setup Enviorment Variables
M2_bronze = None
M3_bronze = None
M3_silver = None
M6_bronze = None
bat = None
bhoref = None
bac = None
bll = None
b = None
M4_silver = None
M4_bronze = None
docsr = None
H_bronze = None
H_silver = None

config = spark.read.option("multiline", "true").json("dbfs:/configs/config.json")
env_name = config.first()["env"].strip().lower()
lz_key = config.first()["lz_key"].strip().lower()
KeyVault_name = f"ingest{lz_key}-meta002-{env_name}"
client_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-ID")
client_secret = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-TENANT-ID")
curated_storage = f"ingest{lz_key}curated{env_name}"
checkpoint_storage = f"ingest{lz_key}xcutting{env_name}"
raw_storage = f"ingest{lz_key}raw{env_name}"
landing_storage = f"ingest{lz_key}landing{env_name}"
external_storage = f"ingest{lz_key}external{env_name}"

spark.conf.set(f"fs.azure.account.auth.type.{curated_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{curated_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{curated_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{curated_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{curated_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

spark.conf.set(f"fs.azure.account.auth.type.{checkpoint_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{checkpoint_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{checkpoint_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{checkpoint_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{checkpoint_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

spark.conf.set(f"fs.azure.account.auth.type.{raw_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{raw_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{raw_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{raw_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{raw_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

spark.conf.set(f"fs.azure.account.auth.type.{landing_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{landing_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{landing_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{landing_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{landing_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

spark.conf.set(f"fs.azure.account.auth.type.{external_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{external_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{external_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{external_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{external_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

bronze_path = f"abfss://bronze@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
silver_path = f"abfss://silver@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
import json

# gold_path = f"abfss://gold@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/{state_under_test}"
# json_location = dbutils.fs.ls(f"{gold_path}/")[-1]
# latest_json_location = json_location.name
# json_path = f"{gold_path}/{latest_json_location}/JSON/"
# json_data = spark.read.format("json").load(json_path)


In [0]:
# ----------------------------------------------------------------------
# C. (Optional, if you can list/read gold paths from this notebook):
#    For each state's gold, check whether the AS-required columns exist.
#    Replace `gold_base` with the path your loader uses, then run.
# ----------------------------------------------------------------------
AS_REQUIRED = [
    # from test_default_mapping_init
    "appealReferenceNumber","paAppealTypePaymentOption",
    "paAppealTypeAipPaymentOption","additionalPaymentInfo",
    # from test_payment_init
    "paymentStatus","rpDcAppealHearingOption","paidDate",
    "paidAmount","paymentDescription",
    # from test_remission_init
    "remissionDecision","remissionDecisionReason","amountRemitted",
    "amountLeftToPay",
]
states = ["appealSubmitted","awaitingRespondentEvidence(a)",
          "awaitingRespondentEvidence(b)","caseUnderReview",
          "reasonsForAppealSubmitted","listing","prepareForHearing",
          "decision","decided(a)","ftpaSubmitted(a)","decided(b)",
          "ftpaSubmitted(b)","ftpaDecided","ended","remitted"]
rows = []
for s in states:
    try:
        gold_path = f"abfss://gold@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/{s}"
        json_location = dbutils.fs.ls(f"{gold_path}/")[-1]
        latest_json_location = json_location.name
        json_path = f"{gold_path}/{latest_json_location}/JSON/"
        jd = spark.read.format("json").load(json_path)
        # jd = load_gold_data(s)   # the same loader your Run_All_States cell uses
        have = set(jd.columns)
        missing = [c for c in AS_REQUIRED if c not in have]
        rows.append((s, len(have), len(missing), ", ".join(missing) or "(all present)"))
    except Exception as e:
        rows.append((s, -1, -1, f"LOAD FAILED: {e!r:.120}"))
import pandas as pd
print(pd.DataFrame(rows, columns=["state","cols_total","missing_count","missing"]).to_string(index=False))